<a id="top"></a>

# Demo: the same loop, driving a real model

```yaml
title: "Demo: the same loop, driving a real model"
keywords: agent loop, run_loop, live model, raw anthropic sdk, multi-step, natural termination, max_steps, sonnet, teacher demo
estimated duration: 12
```

> **Lesson:** L10. Teacher demo — the live counterpart to [L1003](L0703_lecture.ipynb). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: dry-run before class. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` carries a string and cannot represent `tool_use` / `tool_result` blocks. The agent loop is *built on* those blocks, so the live run calls the raw SDK directly. The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. This is the same seam exception L07 made.
>
> The point to land: the loop you built on a stub is the **same loop** that drives a real model. Only the *client* changes — `run_loop` is byte-for-byte the offline version, adapted to read the real SDK's blocks. The model chains two-plus tool calls and terminates naturally.

## Contents

- [1. Setup — the real client, the tools, the schemas](#1-setup--the-real-client-the-tools-the-schemas)
- [2. dispatch — unchanged from the stub demo](#2-dispatch--unchanged-from-the-stub-demo)
- [3. run_loop — the SDK-facing version](#3-run_loop--the-sdk-facing-version)
- [4. Run it — a task that forces two tool calls](#4-run-it--a-task-that-forces-two-tool-calls)
- [5. Takeaways](#5-takeaways)

## 1. Setup — the real client, the tools, the schemas

The raw Anthropic client (key via the config seam), the same `calculator` + `lookup` tools, and — because a real model needs them — JSON-Schema **definitions** for each tool (the loop on a stub didn't need schemas; a real model does). Anchor model: **Claude Sonnet 4.6**.

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=require_anthropic_key())


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the whitelist above blocks names/attributes/calls.
    return str(eval(expression))


POPULATIONS = {
    "tokyo": "37000000",
    "lagos": "15000000",
    "paris": "11000000",
}


def lookup(city: str) -> str:
    """Return a city's population from a tiny in-memory table, or raise KeyError if missing."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# The dispatch table: tool NAME -> the Python function that runs it. The loop
# never hard-codes tool names; it looks each requested name up here.
TOOLS: dict[str, object] = {"calculator": calculator, "lookup": lookup}


# The model only ever sees these DEFINITIONS, never the Python functions.
TOOL_SCHEMAS: list[dict[str, object]] = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression and return the exact result.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "e.g. '17*17 - 1'."}},
            "required": ["expression"],
        },
    },
    {
        "name": "lookup",
        "description": "Look up a city's population from a small internal table.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "a city name, e.g. 'Tokyo'."}},
            "required": ["city"],
        },
    },
]
print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. dispatch — unchanged from the stub demo

The exact `dispatch` from [L1003](L0703_lecture.ipynb): look up the tool, run it, and turn any exception into a `tool_result` with `is_error=True`. Loop-level failure handling does not care whether the model is real or fake.

In [ ]:
def dispatch(tools: dict[str, object], call: object) -> dict[str, object]:
    """Run one requested tool and return a tool_result block.

    On success: a tool_result carrying the tool's output. On ANY exception (unknown
    tool name, the tool raised): a tool_result with is_error=True and a SHORT message
    -- never a traceback. The loop keeps going; the model decides what to do next.
    """
    fn = tools.get(call.name)
    if fn is None:
        return {
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": f"no such tool {call.name!r}",
            "is_error": True,
        }
    try:
        output = fn(**call.input)
        return {"type": "tool_result", "tool_use_id": call.id, "content": output}
    except Exception as exc:
        # repr(exc) is a class name + one line: e.g. KeyError('no population...').
        # Short, descriptive, cheap -- never dump a traceback at the model.
        return {
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": repr(exc),
            "is_error": True,
        }

[↑ Back to top](#top)

## 3. run_loop — the SDK-facing version

The same control flow as the stub demo. The only differences are real-SDK ergonomics: we pass `tools=TOOL_SCHEMAS` (the model needs schemas) and `model=`/`max_tokens=` to the real client. The `tool_use` detection, the message-history invariant, the natural-vs-`max_steps` termination — all identical.

In [ ]:
from dataclasses import dataclass


@dataclass
class RunResult:
    """The answer, the number of model calls, and WHY the loop stopped."""

    final_text: str
    iterations: int
    termination: str


def run_loop(user_msg: str, max_steps: int) -> RunResult:
    """A model -> tool -> model loop against the REAL Anthropic client."""
    messages: list[dict[str, object]] = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        resp = client.messages.create(
            model=MODEL, max_tokens=512, tools=TOOL_SCHEMAS, messages=messages
        )
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            final_text = "".join(b.text for b in resp.content if b.type == "text")
            print(f"[step {step}] natural termination")
            return RunResult(final_text, step, "natural")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for call in tool_uses:
            print(f"[step {step}] tool_use: {call.name}({call.input})")
            results.append(dispatch(TOOLS, call))
        messages.append({"role": "user", "content": results})
    print(f"[stop] max_steps={max_steps} hit")
    return RunResult("", max_steps, "max_steps")

[↑ Back to top](#top)

## 4. Run it — a task that forces two tool calls

A prompt designed to force a **chain**: compute something with `calculator`, then use that result to `lookup` a city's population, then answer. Watch the per-step prints show two-plus tool calls before natural termination.

In [ ]:
task = (
    "Use the calculator to compute 17 squared minus 1, then look up the population of the "
    "city Tokyo, and tell me both numbers. Use the tools; do not answer from memory."
)
result = run_loop(task, max_steps=8)
print()
print("final answer:\n", result.final_text)
print("\niterations :", result.iterations)
print("termination:", result.termination)

[↑ Back to top](#top)

## 5. Takeaways

- The loop is **the same code** as the stub demo — only the client changed. That interchangeability is why we taught the control flow offline first.
- A real model **varies**: the exact tool order and iteration count differ run to run. The loop's job is to be correct regardless — the `max_steps` cap is there precisely because you can't predict the model.
- The per-step `print()` is a **minimum-viable trace** (iteration, tool calls, termination cause). [L11] replaces it with structured tracing on *this exact loop* — keep this code accessible.
- Don't reach for a framework yet: this 40-line loop is the spine LangGraph (L04) will later wrap. Hand-rolling it once is what makes the framework legible.

[↑ Back to top](#top)